# OOP 1: Introduction to Classes and Objects

## Table of Contents

1. [The Problem: Loose Variables Everywhere](#1)
2. [What is a Class?](#2)
3. [Creating an Object](#3)
4. [Instance Attributes and `__init__`](#4)
5. [Methods](#5)
6. [A Realistic Example: The Dataset Class](#6)
7. [Exercise: ModelResult Class](#7)

---
## 1. The Problem: Loose Variables Everywhere <a id="1"></a>

Here's what a typical ML workflow looks like without classes:

```python
X_train, X_test, y_train, y_test = train_test_split(X, y)
feature_names = df.columns.tolist()
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# ... 10 more loose variables ...
```

Now you need to pass this "dataset" to a training function. Which variables do you pass? What if you forget one? What if someone calls them in a different order on a fresh notebook?

A **class** bundles related data and behaviour into one named object. Instead of 8 loose variables, you have one `dataset` object that knows everything about itself.

---
## 2. What is a Class? <a id="2"></a>

A class is a **blueprint** for creating objects. It defines:
- **Attributes**: the data the object holds
- **Methods**: what the object can do

```python
class ClassName:
    # attributes and methods go here
```

Defining a class doesn't create anything yet. Tt's just the blueprint. Creating an object from it is called **instantiation**.

In [ ]:
class Dataset:
    pass  # empty class. The blueprint exists but does nothing

print(Dataset)

---
## 3. Creating an Object <a id="3"></a>

An **object** (also called an **instance**) is created by calling the class. Each object is independent and changing one doesn't affect another.

In [ ]:
ds1 = Dataset()
ds2 = Dataset()

print(ds1)
print(ds2)
print(ds1 is ds2)  # False, they are different objects in memory

In [ ]:
# Python lets you attach attributes to any object at runtime...
ds1.name = "churn_data"
print(ds1.name)

# ...but ds2 doesn't have it
print(ds2.name)

---
## 4. Instance Attributes and `__init__` <a id="4"></a>

Ad-hoc attributes are fragile. The proper way is using `__init__`, a special method Python calls automatically when you create an object. Every instance gets its own copy of the attributes you define there.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

class Dataset:
    def __init__(self, X, y, name= "dataset"):
        self.X = X
        self.y = y
        self.name = name

# Build a small toy dataset
X = pd.DataFrame({"tenure": [12, 5, 24, 3], "monthly_charges": [70, 30, 90, 20]})
y = pd.Series([1, 0, 1, 0], name="churn")

ds = Dataset(X, y, name="churn_demo")
print(ds.name)
print(ds.X.shape)
print(ds.y.values)

---
## 5. Methods <a id="5"></a>

Methods are functions defined inside a class. The first parameter is always `self`, which refers to the specific object the method is being called on.

Methods can both **read** and **modify** the object's attributes.

In [ ]:
class Dataset:
    def __init__(self, X: pd.DataFrame, y: pd.Series, name = "dataset"):
        self.X = X
        self.y = y
        self.name = name
        self._splits = {}  # will be filled by split()

    def split(self, test_size = 0.2, random_state = 42):
        X_tr, X_te, y_tr, y_te = train_test_split(
            self.X, self.y, test_size=test_size,
            random_state=random_state, stratify=self.y
        )
        self._splits = {
            "X_train": X_tr, "X_test": X_te,
            "y_train": y_tr, "y_test": y_te,
        }
        return self

    def summary(self):
        n, p = self.X.shape
        pos = int(self.y.sum())
        return (
            f"Dataset '{self.name}': {n} rows, {p} features, "
            f"{pos}/{n} positive labels ({100*pos/n:.1f}%)"
        )

X = pd.DataFrame({"tenure": [12, 5, 24, 3, 8, 17, 22, 1],
                  "monthly_charges": [70, 30, 90, 20, 55, 80, 95, 15]})
y = pd.Series([1, 0, 1, 0, 0, 1, 1, 0], name="churn")

ds = Dataset(X, y, name="churn_demo")
print(ds.summary())

ds.split(test_size=0.25)
print("Train rows:", ds._splits["X_train"].shape[0])
print("Test  rows:", ds._splits["X_test"].shape[0])

### Properties

A `@property` lets you expose a computed value as if it were an attribute: no `()` needed when you access it.

In [ ]:
class Dataset:
    def __init__(self, X, y, name="dataset"):
        self.X = X
        self.y = y
        self.name = name
        self._splits = {}

    def split(self, test_size=0.2, random_state=42):
        X_tr, X_te, y_tr, y_te = train_test_split(
            self.X, self.y, test_size=test_size,
            random_state=random_state, stratify=self.y
        )
        self._splits = {"X_train": X_tr, "X_test": X_te,
                        "y_train": y_tr, "y_test": y_te}
        return self

    @property
    def train(self):
        if not self._splits:
            raise RuntimeError("Call .split() first.")
        return self._splits["X_train"], self._splits["y_train"]

    @property
    def test(self):
        if not self._splits:
            raise RuntimeError("Call .split() first.")
        return self._splits["X_test"], self._splits["y_test"]

X = pd.DataFrame({"tenure": [12, 5, 24, 3, 8, 17, 22, 1],
                  "monthly_charges": [70, 30, 90, 20, 55, 80, 95, 15]})
y = pd.Series([1, 0, 1, 0, 0, 1, 1, 0], name="churn")

ds = Dataset(X, y).split()

X_train, y_train = ds.train   # clean property access, no dict indexing
X_test,  y_test  = ds.test
print("Train:", X_train.shape, "| Test:", X_test.shape)

---
## 6. A Realistic Example: The Dataset Class <a id="6"></a>

Let's build the full `Dataset` class we'll use throughout Day 1.

In [ ]:
class Dataset:
    def __init__(self, X, y, name="dataset"):
        self.X = X
        self.y = y
        self.name = name
        self._splits = {}

    # ------------------------------------------------------------------ #
    # Core operations
    # ------------------------------------------------------------------ #

    def split(self, test_size, random_state=42):
        X_tr, X_te, y_tr, y_te = train_test_split(
            self.X, self.y, test_size=test_size,
            random_state=random_state, stratify=self.y
        )
        self._splits = {"X_train": X_tr, "X_test": X_te,
                        "y_train": y_tr, "y_test": y_te}
        return self

    # ------------------------------------------------------------------ #
    # Properties
    # ------------------------------------------------------------------ #

    @property
    def train(self):
        self._require_split()
        return self._splits["X_train"], self._splits["y_train"]

    @property
    def test(self):
        self._require_split()
        return self._splits["X_test"], self._splits["y_test"]

    @property
    def feature_names(self):
        return self.X.columns.tolist()

    @property
    def n_samples(self):
        return len(self.X)

    # ------------------------------------------------------------------ #
    # Reporting
    # ------------------------------------------------------------------ #

    def summary(self):
        n, p = self.X.shape
        pos = int(self.y.sum())
        return (
            f"Dataset '{self.name}': {n} rows, {p} features, "
            f"{pos}/{n} positive labels ({100*pos/n:.1f}%)"
        )

    def class_balance(self) -> pd.Series:
        return self.y.value_counts(normalize=True).rename("proportion")

    # ------------------------------------------------------------------ #
    # Internal helpers
    # ------------------------------------------------------------------ #

    def _require_split(self) -> None:
        if not self._splits:
            raise RuntimeError("Call .split() before accessing train/test data.")


# --- Demo ---
np.random.seed(42)
X_demo = pd.DataFrame({
    "tenure":          np.random.randint(1, 60, 200),
    "monthly_charges": np.random.uniform(20, 100, 200),
    "num_products":    np.random.randint(1, 5, 200),
})
y_demo = pd.Series((X_demo["tenure"] < 10).astype(int), name="churn")

ds = Dataset(X_demo, y_demo, name="telecom_churn").split(test_size=0.2)
print(ds.summary())
print()
print("Features:", ds.feature_names)
print()
print("Class balance:")
print(ds.class_balance())

---
## 7. Exercise: ModelResult Class <a id="7"></a>

When you train a model and evaluate it, you get a bunch of loose objects: `model`, `preds`, `probs`, `score`, `feature_importances_`, ...

Create a `ModelResult` class that bundles these together.

**Requirements:**
- `__init__` accepts: `model`, `X_test`, `y_test`, `model_name`
- Computes and stores `predictions` and `probabilities` in `__init__`
- `report()` prints a sklearn classification report  
- `feature_importance()` returns a sorted `pd.Series` if the model has `.feature_importances_`, otherwise raises `AttributeError` with a helpful message

In [ ]:
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier

# Your implementation here
class ModelResult:
    pass


# --- Test your implementation ---
# (Assumes ds from the cell above is in scope)
X_train, y_train = ds.train
X_test,  y_test  = ds.test

clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)

result = ModelResult(clf, X_test, y_test, model_name="RandomForest")
result.report()
print()
print(result.feature_importance())

Do you need a bit of help? Here you can find a possible implementation of the class.

> <details>
> <summary>Click here</summary>
>
> ```python
> class ModelResult:
>     def __init__(self, model, X_test, y_test, model_name: str = "model"):
>         self.model = model
>         self.model_name = model_name
>         self.X_test = X_test
>         self.y_test = y_test
>         self.predictions = model.predict(X_test)
>         self.probabilities = (
>             model.predict_proba(X_test)[:, 1]
>             if hasattr(model, "predict_proba") else None
>         )
>
>     def report(self) -> None:
>         print(f"=== {self.model_name} ===")
>         print(classification_report(self.y_test, self.predictions))
>
>     def feature_importance(self) -> pd.Series:
>         if not hasattr(self.model, "feature_importances_"):
>             raise AttributeError(
>                 f"{type(self.model).__name__} does not expose feature_importances_."
>             )
>         return (
>             pd.Series(self.model.feature_importances_,
>                       index=self.X_test.columns)
>             .sort_values(ascending=False)
>         )
> ```
> </details>